# CatBoost with three random seeds
Run all cells from the project directory. Compare seed 42 with a fixed average of seeds 42, 73 and 101. Folds, features and parameters stay fixed so only seed averaging changes. The exported CSV uses a 0.5 threshold.

In [1]:
%pip install catboost pandas scikit-learn

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
from pathlib import Path
import numpy as np
import pandas as pd
from catboost import CatBoostClassifier
from sklearn.model_selection import StratifiedKFold

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'data' / 'raw').is_dir())  # project root
DATA = ROOT / 'data' / 'raw'
SUBMISSIONS = ROOT / 'submissions'
train = pd.read_csv(DATA / 'train.csv')
test = pd.read_csv(DATA / 'test.csv')
sample = pd.read_csv(DATA / 'sample_submission.csv')
y = train.pop('Transported').astype(int)
rows = pd.concat([train, test], ignore_index=True)
assert sample.PassengerId.equals(test.PassengerId) and rows.PassengerId.is_unique

## Passenger and cabin features
The features use cabin occupancy, age and cabin bands, spending bands, and passenger groups. Group counts and unanimous group attributes use the complete unlabeled passenger manifest. Deck and side are used directly, not as deck transport rates, because rates calculated before splitting for validation would expose validation labels.

In [3]:
SPEND = ["RoomService", "FoodCourt", "ShoppingMall", "Spa", "VRDeck"]


def features(rows):
    x = rows.copy()
    group = x.pop("PassengerId").str.split("_").str[0]
    family = x.pop("Name").str.split().str[-1]
    x[["Deck", "Number", "Side"]] = x.pop("Cabin").str.split("/", expand=True)
    x["Number"] = pd.to_numeric(x.Number)
    x["GroupNumber"] = group.astype(int)
    x["GroupSize"] = group.groupby(group).transform("size")
    x["FamilySize"] = family.groupby(family).transform("size")
    for col in ["Deck", "Side", "Number", "HomePlanet"]:
        matches = x[col].groupby(group)
        unanimous = matches.transform("first").where(matches.transform("nunique") == 1)
        x[col] = x[col].fillna(unanimous)
    x["TotalSpend"] = x[SPEND].sum(axis=1, min_count=1)
    x["MissingSpend"] = x[SPEND].isna().sum(axis=1)
    x["GroupMeanSpend"] = x.TotalSpend.groupby(group).transform("mean")
    x.loc[x.CryoSleep.eq(True), SPEND] = x.loc[x.CryoSleep.eq(True), SPEND].fillna(0)
    for col in ["Deck", "Side", "HomePlanet", "Destination", "CryoSleep", "VIP"]:
        x[col] = x[col].astype("string").fillna("Unknown").astype("category")
    return x

In [4]:
x = features(rows)
x['CabinMembers'] = rows.groupby('Cabin').PassengerId.transform('size')
for source, cuts in {'Age': [17, 43], 'Number': [300, 690, 1170],
                     'GroupNumber': [3400, 7300], 'TotalSpend': [1, 2250]}.items():
    x[source + 'Band'] = pd.cut(x[source], [-np.inf, *cuts, np.inf], labels=False)



In [5]:
folds = list(StratifiedKFold(5, shuffle=True, random_state=42).split(x.iloc[:len(y)], y))

In [6]:
def evaluate(inputs, seed):
    predictions = np.full(len(y), np.nan)
    test_predictions = []
    for fold, (tr, va) in enumerate(folds, start=1):
        model = CatBoostClassifier(iterations=2000, depth=6, learning_rate=.01,
                    cat_features=inputs.select_dtypes('category').columns.tolist(),
                    thread_count=4, verbose=False, random_seed=seed, allow_writing_files=False)
        model.fit(inputs.iloc[tr], y.iloc[tr])
        predictions[va] = model.predict_proba(inputs.iloc[va])[:, 1]
        test_predictions.append(model.predict_proba(inputs.iloc[len(y):])[:, 1])
        print('Finished fold', fold, flush=True)
    assert np.isfinite(predictions).all()
    return predictions, np.mean(test_predictions, axis=0)

results = [evaluate(x, seed) for seed in (42, 73, 101)]
seed42_oof, seed42_test = results[0]
seed_oof = np.mean([oof for oof, _ in results], axis=0)
seed_test = np.mean([test for _, test in results], axis=0)

Finished fold 1
Finished fold 2
Finished fold 3
Finished fold 4
Finished fold 5
Finished fold 1
Finished fold 2
Finished fold 3
Finished fold 4
Finished fold 5
Finished fold 1
Finished fold 2
Finished fold 3
Finished fold 4
Finished fold 5


In [7]:
correct = pd.DataFrame({'Seed 42': seed42_oof, 'Seed average': seed_oof}).ge(.5).eq(y, axis=0)
scores = pd.DataFrame([correct.iloc[va].mean() for _, va in folds], index=range(1, 6))
scores.loc['Overall'] = correct.mean()
scores['Change'] = scores['Seed average'] - scores['Seed 42']
display(scores.round(5))
assert np.isfinite(seed_test).all()
output = sample.copy()
output['Transported'] = seed_test >= .5
path = SUBMISSIONS / 'submission_catboost_seeds.csv'
output.to_csv(path, index=False)
print(path)
print('Predictions changed by averaging:', ((seed42_test >= .5) != output.Transported).sum())

,Seed 42,Seed average,Change
1,0.81944,0.81886,-0.00058
2,0.80564,0.80736,0.00173
3,0.81311,0.82116,0.00805
4,0.81818,0.81646,-0.00173
5,0.80437,0.80150,-0.00288
Overall,0.81215,0.81307,0.00092


c:\Users\shaya\titanic-spaceShip\submissions\submission_catboost_seeds.csv
Predictions changed by averaging: 23
